In [1]:
import numpy as np
import pandas as pd
import json

In [2]:
sales_orders = pd.read_csv('datasets/flatten-the-stack/sales_orders.csv', parse_dates=['order_date'])

In [3]:
sales_orders

,order_number,order_date,line_items,fulfillment
0,387005,2016-01-22,"[\r\n {\r\n ""product"": {\r\n ""product...",In store
1,395005,2016-01-30,"[\r\n {\r\n ""product"": {\r\n ""product...",In store
2,423011,2016-02-27,"[\r\n {\r\n ""product"": {\r\n ""product...",In store
3,600006,2016-08-22,"[\r\n {\r\n ""product"": {\r\n ""product...",Online
4,652002,2016-10-13,"[\r\n {\r\n ""product"": {\r\n ""product...",In store
...,...,...,...,...
95,2144004,2020-11-13,"[\r\n {\r\n ""product"": {\r\n ""product...",In store
96,2152009,2020-11-21,"[\r\n {\r\n ""product"": {\r\n ""product...",Online
97,2165005,2020-12-04,"[\r\n {\r\n ""product"": {\r\n ""product...",In store
98,2175000,2020-12-14,"[\r\n {\r\n ""product"": {\r\n ""product...",In store


In [4]:
sales_orders['line_items'] = sales_orders['line_items'].apply(json.loads)

In [5]:
tmp = sales_orders.explode('line_items', ignore_index=False)
tmp['line_item'] = tmp.groupby(level=0).cumcount() + 1
tmp = tmp.reset_index(drop=True)
tmp

,order_number,order_date,line_items,fulfillment,line_item
0,387005,2016-01-22,{'product': {'product_name': 'MGS Age of Empir...,In store,1
1,387005,2016-01-22,{'product': {'product_name': 'A. Datum Bridge ...,In store,2
2,387005,2016-01-22,{'product': {'product_name': 'WWI Desktop PC1....,In store,3
3,395005,2016-01-30,{'product': {'product_name': 'Contoso DVD 60 D...,In store,1
4,395005,2016-01-30,{'product': {'product_name': 'SV DVD 38 DVD St...,In store,2
...,...,...,...,...,...
225,2175000,2020-12-14,{'product': {'product_name': 'The Phone Compan...,In store,1
226,2175000,2020-12-14,{'product': {'product_name': 'The Phone Compan...,In store,2
227,2175000,2020-12-14,{'product': {'product_name': 'Fabrikam Laptop1...,In store,3
228,2175000,2020-12-14,{'product': {'product_name': 'SV DVD 14-Inch P...,In store,4


In [6]:
tmp[['quantity', 'product_name', 'product_price']] = pd.json_normalize(tmp['line_items'])

result = tmp[[
    'order_number', 'order_date',
    'line_item',
    'product_name', 'product_price', 'quantity',
    'fulfillment'
]]
result

,order_number,order_date,line_item,product_name,product_price,quantity,fulfillment
0,387005,2016-01-22,1,MGS Age of Empires II Gold Edition2009 E172,32.00,3,In store
1,387005,2016-01-22,2,A. Datum Bridge Digital Camera M300 Pink,186.90,2,In store
2,387005,2016-01-22,3,WWI Desktop PC1.80 E1801 Silver,269.90,1,In store
3,395005,2016-01-30,1,Contoso DVD 60 DVD Storage Binder L20 Black,22.89,5,In store
4,395005,2016-01-30,2,SV DVD 38 DVD Storage Binder E25 Silver,9.99,5,In store
...,...,...,...,...,...,...,...
225,2175000,2020-12-14,1,The Phone Company Smart phones without camera ...,129.00,2,In store
226,2175000,2020-12-14,2,The Phone Company PDA Phone Unlocked 3.7 inche...,238.00,4,In store
227,2175000,2020-12-14,3,Fabrikam Laptop10.1 M0101 Silver,384.90,3,In store
228,2175000,2020-12-14,4,SV DVD 14-Inch Player Portable L100 White,259.99,4,In store


## What is the sum of total online sales? (round to nearest integer)

In [7]:
online = result[result['fulfillment'] == 'Online']
(online['product_price'] * online['quantity']).sum().round()

18239.0